# Final system on the 40 dev videos: OSTrack + V-JEPA encoder + V-JEPA predictor (Kaggle, self-contained)

`final` = the best variant (V-JEPA re-localization verified by OSTrack, score >= 0.6) with
1. **predictor occlusion warning** (v2, ranks, top 5%, 16 frames): freezes the target memory when the
   target starts to go behind something,
2. **memory guard**: the memory is only updated when V-JEPA agrees the box still looks like the first
   appearance (half of the occlusions were lost BEFORE they started, on a confident distractor),
3. **drift check**: candidates also when OSTrack is confident but V-JEPA sees the target clearly elsewhere,
4. **motion**: candidates on the target's predicted path are preferred (identical objects),
5. **predictor future candidates**: while warned, where the predicted future frame shows the target.

Each run removes or swaps one part, so its contribution is measured; `final_no_predictor` is the same
system with the warning from OSTrack's score only and no future candidates - the predictor's contribution.

| run | | time |
|---|---|---|
| cache + surprise maps | | ~2 h |
| final | everything | ~45 min |
| final_no_predictor | warning from the score only, no future candidates | ~30 min |
| final_no_future | without 5 | ~30 min |
| final_no_warning | without 1 (and 5) | ~30 min |
| final_no_guard | without 2 | ~45 min |
| final_no_drift | without 3 | ~45 min |
| final_no_motion | without 4 | ~45 min |
| guard_drift | only 2 + 3 (no predictor, no motion) | ~30 min |

Output: `/kaggle/working/final.zip`. Settings: **Accelerator = GPU T4 x2**, **Internet = On**, no inputs. Then **Save Version -> Save & Run All**.

In [ ]:
import socket
try:
    socket.create_connection(('github.com', 443), timeout=10).close()
except OSError:
    raise SystemExit('NO INTERNET: Settings (right panel) -> Internet = On, then run again.')
import torch
if torch.cuda.device_count() == 0:
    raise SystemExit('NO GPU: Settings (right panel) -> Accelerator = GPU T4 x2, then run again.')
print(torch.cuda.device_count(), 'GPUs:', torch.cuda.get_device_name(0))

In [ ]:
import subprocess
subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/MariiaHamanuk/Object-Tracking.git',
                '/kaggle/working/Object-Tracking'], check=True)
!pip install -q timm==0.6.13 easydict visdom tensorboardX jpeg4py lmdb gdown "transformers>=4.53"
!gdown 1dySfPrSk-knAEo0lTXqZ1kMk3BCvVofV -O /kaggle/working/OSTrack_ep0300.pth.tar

In [ ]:
import os, glob, subprocess, sys, numpy as np
os.chdir('/kaggle/working/Object-Tracking')
sys.path.insert(0, '.')
os.environ['HF_HUB_DISABLE_XET'] = '1'
os.makedirs('/kaggle/tmp', exist_ok=True)
W = '/kaggle/working'
CKPT = f'{W}/OSTrack_ep0300.pth.tar'
CACHE = '/kaggle/tmp/cache/vjepa'
SPLIT, EVERY, ZIP = 'configs/splits/dev_occlusion.txt', 8, 'final.zip'
N_VIDEOS = sum(1 for s in open(SPLIT) if s.strip())
BEST = ['--reloc-verify', '3', '--verify-min', '0.6']    # the best dev variant
N_GPU = max(torch.cuda.device_count(), 1)
os.makedirs(f'{W}/logs', exist_ok=True)

def run_shards(name, args):
    procs = []
    for i in range(N_GPU):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i))
        log = open(f'{W}/logs/{name}_shard{i}.log', 'w')
        procs.append(subprocess.Popen(['python'] + args + ['--shard', str(i), '--num-shards', str(N_GPU)],
                                      env=env, stdout=log, stderr=subprocess.STDOUT))
    codes = [p.wait() for p in procs]
    os.system(f"grep -hE 'FAILED|Error|Traceback' {W}/logs/{name}_shard*.log | head -n 20")
    os.system(f"tail -n 1 {W}/logs/{name}_shard*.log")
    print(name, 'exit codes', codes)

def pack():
    os.system(f'cd {W} && rm -f {ZIP} && zip -qr {ZIP} results_raw logs 2>/dev/null; ls -la {W}/{ZIP}')

def track(name, extra):
    """The best variant + `extra` on SPLIT -> results_raw/<name>/lasot."""
    run_shards(name, ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', SPLIT, '--vjepa-cache', CACHE,
                      '--out', f'{W}/results_raw/{name}/lasot', '--work', '/kaggle/tmp'] + BEST + extra)
    pack()

## Self-test

In [ ]:
# 30 frames, random V-JEPA maps: every warning mode, the online predictor and future candidates run
os.makedirs('/kaggle/tmp/selftest/cache', exist_ok=True)
np.savez('/kaggle/tmp/selftest/cache/airplane-1.npz', frames=np.arange(0, 3000, EVERY),
         maps=np.random.randn(len(range(0, 3000, EVERY)), 16, 16, 1024).astype(np.float16),
         surprise=np.random.rand(len(range(0, 3000, EVERY)), 16, 16).astype(np.float16))
base = ['scripts/run_ostrack_lasot.py', '--checkpoint', CKPT, '--sequences', 'airplane-1', '--max-frames', '30',
        '--vjepa-cache', '/kaggle/tmp/selftest/cache', '--work', '/kaggle/tmp/selftest/work'] + BEST
for name, extra in [('off', []), ('score', ['--occlusion-warning', 'score']), ('both', ['--occlusion-warning', 'both']),
                    ('rise', ['--occlusion-warning', 'rise', '--warn-relax']),
                    ('online', ['--occlusion-warning', 'online', '--online-predictor', '--future-candidates', '2'])]:
    subprocess.run(['python'] + base + ['--out', f'/kaggle/tmp/selftest/{name}'] + extra, check=True)
import pandas as pd
log = pd.read_csv('/kaggle/tmp/selftest/online/airplane-1_reloc.csv')
assert log.online.notna().all(), 'online predictor surprise missing'
print('SELF-TEST OK')

In [ ]:
# the final system end to end on 30 frames
subprocess.run(['python'] + base + ['--out', '/kaggle/tmp/selftest/final', '--occlusion-warning', 'both',
                '--memory-guard', '0.2', '--drift-check', '--motion', '0.2', '--online-predictor',
                '--future-candidates', '2'], check=True)
print('FINAL SELF-TEST OK')

## V-JEPA cache with the predictor's surprise maps

In [ ]:
for attempt in range(2):  # a re-run only does the videos that failed to download
    run_shards('vjepa_cache', ['scripts/cache_vjepa.py', '--sequences', SPLIT, '--out', CACHE, '--work', '/kaggle/tmp',
                               '--every', str(EVERY), '--surprise', '--surprise-out', f'{W}/results_raw/surprise_maps'])
    n = len(glob.glob(f'{CACHE}/*.npz'))
    print(n, 'of', N_VIDEOS, 'videos cached')
    if n == N_VIDEOS:
        break
assert n == N_VIDEOS, 'V-JEPA cache incomplete'
pack()

## Runs

In [ ]:
GUARD, DRIFT, MOTION = ['--memory-guard', '0.2'], ['--drift-check'], ['--motion', '0.2']
WARN, FUTURE = ['--occlusion-warning', 'both'], ['--online-predictor', '--future-candidates', '2']
RUNS = [('final', WARN + GUARD + DRIFT + MOTION + FUTURE),
        ('final_no_predictor', ['--occlusion-warning', 'score'] + GUARD + DRIFT + MOTION),
        ('final_no_future', WARN + GUARD + DRIFT + MOTION),
        ('final_no_warning', GUARD + DRIFT + MOTION),
        ('final_no_guard', WARN + DRIFT + MOTION + FUTURE),
        ('final_no_drift', WARN + GUARD + MOTION + FUTURE),
        ('final_no_motion', WARN + GUARD + DRIFT + FUTURE),
        ('guard_drift', GUARD + DRIFT)]
for name, extra in RUNS:
    track(f'ostrack_vjepa_{name}', extra)
os.system(f'rm -rf {W}/Object-Tracking {CKPT}')